# Testing the documentation extraction pipeline

This notebook imports [`dataset/buildDataset/build.py`](dataset/buildDataset/build.py)
directly and calls its actual functions (`find_documentation_header`,
`extract_documentation`, `strip_comments`, `tokenize`, `calculate_entropy`,
`doc_redundancy`, `doc_code_overlap`) on test strings, so the exact code used
for the paper's mining pipeline is what gets exercised here -- no hand-typed
copies.

The only function defined locally is `clean_comments`, ported from
[`dataset/buildDataset/combine.ipynb`](dataset/buildDataset/combine.ipynb) --
that one can't be imported since it lives in a notebook, not a `.py` module.

## 1. Import `build.py` directly

In [1]:
import os
import sys
import re
import textwrap

import numpy as np
import textstat

# build.py raises ValueError at import time unless a GITHUB_TOKEN_* env var is set.
# That check happens before any network call, so a dummy value satisfies it without
# contacting GitHub. None of the functions used below (tokenize, calculate_entropy,
# doc_redundancy, doc_code_overlap, strip_comments, find_documentation_header,
# extract_documentation) touch the network, git, or the filesystem beyond this.
os.environ.setdefault("GITHUB_TOKEN_1", "dummy-token-for-local-testing")

BUILD_DIR = os.path.join(os.getcwd(), "dataset", "buildDataset")
sys.path.insert(0, BUILD_DIR)

import build

print("Imported build.py successfully -- using its functions directly below.")


GitHub token loaded successfully.
Imported build.py successfully -- using its functions directly below.


## 2. `clean_comments` (from `combine.ipynb`) -- the one function that can't be imported

In [2]:
def clean_comments(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # Single alternation scanned left-to-right (finditer) instead of three
    # independent findall passes over the same text. Three separate passes
    # can double-count: if a "#"/"//" comment's content happens to contain
    # something that looks like a \"\"\"...\"\"\" span (or vice versa), the
    # old code matched it in both passes and appended it twice. Scanning
    # once with one combined pattern means each character position is
    # consumed by at most one match, so overlapping patterns can't recur.
    #
    # The inline-comment alternative also stops at a following triple-quote
    # or /* block, not just the next #/// or end of string -- otherwise a
    # "#" comment sitting right next to a standalone docstring fragment
    # (both flattened into one doc_text string by " ".join(doc_list))
    # swallows that adjacent fragment whole, quote markers included,
    # instead of letting it be independently matched and stripped.
    combined_pattern = re.compile(
        r'/\*+([\s\S]*?)\*/'                                    # C-style block comments
        r'|["\']{3}([\s\S]*?)["\']{3}'                           # triple-quoted strings/docstrings
        r'|(?:#|///|//)\s*(.*?)\s*(?=#|///|//|["\']{3}|/\*|$)'    # inline comments
    )

    cleaned_comments = []
    for m in combined_pattern.finditer(text):
        item = next(g for g in m.groups() if g is not None)

        # Remove the leading '*' from each line (common in JSDoc/C-style)
        clean_item = re.sub(r'^\s*\* ?', '', item, flags=re.MULTILINE)

        # Collapse newlines and tabs into a single space
        clean_item = ' '.join(clean_item.split())

        if clean_item.strip():
            cleaned_comments.append(clean_item.strip())

    return ' '.join(cleaned_comments)


## 3. Test strings

In [3]:


test_code_js = '''/**
 * Computes the factorial of n using recursion.
 * @param {number} n - non-negative integer
 * @returns {number} factorial of n
 */
function factorial(n) {
    // base case
    if (n <= 1) return 1;
    return n * factorial(n - 1); // recursive case
}
'''
# ---------------------------------------------------------------- C# (.cs)
test_code_cs = '''/// <summary>
/// Computes the factorial of n using recursion.
/// </summary>
/// <param name="n">non-negative integer</param>
/// <returns>factorial of n</returns>
public static long Factorial(int n)
{
    // base case
    if (n <= 1) return 1;
    return n * Factorial(n - 1); // recursive case
}
'''
 
# ------------------------------------------------------------- C / C++ (.c, .cpp, .h, ...)
test_code_c = '''/**
 * Computes the factorial of n using recursion.
 * @param n non-negative integer
 * @return factorial of n
 */
long factorial(int n) {
    // base case
    if (n <= 1) return 1;
    return n * factorial(n - 1); // recursive case
}
'''
 
# ---------------------------------------------------------------- Go (.go)
test_code_go = '''// Factorial computes the factorial of n using recursion.
// n must be a non-negative integer.
func Factorial(n int) int {
    // base case
    if n <= 1 {
        return 1
    }
    return n * Factorial(n-1) // recursive case
}
'''
 
# -------------------------------------------------------------- Java (.java)
test_code_java = '''/**
 * Computes the factorial of n using recursion.
 * @param n non-negative integer
 * @return factorial of n
 */
public static long factorial(int n) {
    // base case
    if (n <= 1) return 1;
    String html = """
              <html>
                  <body>
                      <p>Hello, World!</p>
                  </body>
              </html>
              """;
    return n * factorial(n - 1); // recursive case
}
'''
 
# ------------------------------------------------------ JavaScript (.js, .jsx)
test_code_js = '''/**
 * Computes the factorial of n using recursion.
 * @param {number} n - non-negative integer
 * @returns {number} factorial of n
 */
function factorial(n) {
    // base case
    if (n <= 1) return 1;
    return n * factorial(n - 1); // recursive case
}
'''
 
# ---------------------------------------------------- Kotlin (.kt, .kts)
test_code_kt = '''/**
 * Computes the factorial of n using recursion.
 * @param n non-negative integer
 * @return factorial of n
 */
fun factorial(n: Int): Long {
    // base case // rakakaka
    if (n <= 1) return 1
    return n * factorial(n - 1) // recursive case
}
'''
 
# ---------------------------------------------------------------- PHP (.php)
test_code_php = '''/**
 * Computes the factorial of n using recursion.
 * @param int $n non-negative integer
 * @return int factorial of n
 */
function factorial($n) {
    # base case
    if ($n <= 1) return 1;
    return $n * factorial($n - 1); // recursive case
}
'''
 
# ------------------------------------------------------------- Python (.py)
test_code_py = '''def factorial(n):
    """
    Computes the factorial of n using recursion.
    :param n: non-negative integer
    :return: factorial of n
    """
    x = 1 # """lukas is the best"""

    test = """this better not be in output"""
    """this better be in the output"""
    #this too
    # base case1
    if n <= 1:
    """ bing inner blam"""
        return 1
    return n * factorial(n - 1)  # recursive case
'''
 
# ---------------------------------------------------------- Scala (.scala)
test_code_scala = '''/**
 * Computes the factorial of n using recursion.
 * @param n non-negative integer
 * @return factorial of n
 */
def factorial(n: Int): Long = {
    // base case
    if (n <= 1) 1
    else n * factorial(n - 1) // recursive case
}
'''
 
# ---------------------------------------------------------- Swift (.swift)
test_code_swift = '''/// Computes the factorial of n using recursion.
/// - Parameter n: non-negative integer
/// - Returns: factorial of n
func factorial(_ n: Int) -> Int {
    // base case
    if n <= 1 { return 1 }
    return n * factorial(n - 1) // recursive case
}
'''
 
print(test_code_py)
print("========================================")
print(test_code_js)


def factorial(n):
    """
    Computes the factorial of n using recursion.
    :param n: non-negative integer
    :return: factorial of n
    """
    x = 1 # """lukas is the best"""

    test = """this better not be in output"""
    """this better be in the output"""
    #this too
    # base case1
    if n <= 1:
    """ bing inner blam"""
        return 1
    return n * factorial(n - 1)  # recursive case

/**
 * Computes the factorial of n using recursion.
 * @param {number} n - non-negative integer
 * @returns {number} factorial of n
 */
function factorial(n) {
    // base case
    if (n <= 1) return 1;
    return n * factorial(n - 1); // recursive case
}



## 4. Run `build.find_documentation_header` + `build.extract_documentation` on every language sample, and verify each one's documentation was captured correctly

In [4]:
factorial_samples = {
    "Python":     (test_code_py,    1, ".py"),
    "JavaScript": (test_code_js,    6, ".js"),
    "C#":         (test_code_cs,    6, ".cs"),
    "C/C++":      (test_code_c,     6, ".c"),
    "Go":         (test_code_go,    3, ".go"),
    "Java":       (test_code_java,  6, ".java"),
    "Kotlin":     (test_code_kt,    6, ".kt"),
    "PHP":        (test_code_php,   6, ".php"),
    "Scala":      (test_code_scala, 6, ".scala"),
    "Swift":      (test_code_swift, 4, ".swift"),
}

must_contain = ["factorial of n using recursion", "base case", "recursive case"]

extraction_results = {}
all_passed = True

for lang, (code, func_line, ext) in factorial_samples.items():
    lines = code.splitlines()
    adj_start = build.find_documentation_header(lines, func_line, ext)
    doc_list, _ = build.extract_documentation(lines, adj_start, len(lines), ext)
    doc_text = " ".join(doc_list)

    raw_code = "\n".join(lines[adj_start - 1:len(lines)])
    code_text = textwrap.dedent(raw_code)
    code_text_no_doc = build.strip_comments(code_text, ext)
    print(code_text_no_doc)
    missing = [s for s in must_contain if s not in doc_text]
    ok = not missing
    all_passed &= ok

    extraction_results[lang] = {
        "doc_text": doc_text,
        "code_text_no_doc": code_text_no_doc,
    }

    print(f"=== {lang} ({ext}) -- {'PASS' if ok else 'FAIL'} ===")
    print(f"function line: {func_line} | adjusted doc start: {adj_start}")
    print("doc_text:", repr(doc_text))
    if missing:
        print(f"  MISSING expected content: {missing}")
    print()

print("ALL LANGUAGES CAPTURED THEIR DOCUMENTATION CORRECTLY" if all_passed else "SOME LANGUAGES FAILED")

est_code_py = '''def factorial(n):
    """
    Computes the factorial of n using recursion.
    :param n: non-negative integer
    :return: factorial of n
    """
    x = 1 # """lukas is the best"""

    test = """this better not be in output"""
    """this better be in the output"""
    #this too
    # base case1
    if n <= 1:
    """ bing inner blam"""
        return 1
    return n * factorial(n - 1)  # recursive case
'''

def factorial(n):
    x = 1
    test = """this better not be in output"""
    if n <= 1:
        return 1
    return n * factorial(n - 1)
=== Python (.py) -- PASS ===
function line: 1 | adjusted doc start: 1
doc_text: '"""\n    Computes the factorial of n using recursion.\n    :param n: non-negative integer\n    :return: factorial of n\n    """ # """lukas is the best""" """this better be in the output""" #this too # base case1 """ bing inner blam""" # recursive case'

function factorial(n) {
    if (n <= 1) return 1;
    return n * factorial(n - 1);
}
=== JavaScript (.js) -- PASS ===
function line: 6 | adjusted doc start: 1
doc_text: '/**\n * Computes the factorial of n using recursion.\n * @param {number} n - non-negative integer\n * @returns {number} factorial of n\n */ // base case // recursive case'

public static long Factorial(int n)
{
    if (n <= 1) return 1;
    return n * Factorial(n - 1);
}
=== C# (.cs) -- PASS ===
function line: 6 | adjusted doc start: 1
doc_text: '/// <sum

## 5. Apply `clean_comments` (from `combine.ipynb`) to each language's extracted text

In [5]:
for lang, result in extraction_results.items():
    cleaned = clean_comments(result["doc_text"])
    result["doc_cleaned"] = cleaned
    print(f"--- {lang} -- cleaned doc text ---")
    print(cleaned)
    print()

    est_code_py = '''def factorial(n):
    """
    Computes the factorial of n using recursion.
    :param n: non-negative integer
    :return: factorial of n
    """
    x = 1 # """lukas is the best"""

    test = """this better not be in output"""
    """this better be in the output"""
    #this too
    # base case1
    if n <= 1:
    """ bing inner blam"""
        return 1
    return n * factorial(n - 1)  # recursive case
'''

--- Python -- cleaned doc text ---
Computes the factorial of n using recursion. :param n: non-negative integer :return: factorial of n lukas is the best this better be in the output this too base case1 bing inner blam recursive case

--- JavaScript -- cleaned doc text ---
Computes the factorial of n using recursion. @param {number} n - non-negative integer @returns {number} factorial of n base case recursive case

--- C# -- cleaned doc text ---
<summary> Computes the factorial of n using recursion. </summary> <param name="n">non-negative integer</param> <returns>factorial of n</returns> base case recursive case

--- C/C++ -- cleaned doc text ---
Computes the factorial of n using recursion. @param n non-negative integer @return factorial of n base case recursive case

--- Go -- cleaned doc text ---
Factorial computes the factorial of n using recursion. n must be a non-negative integer. base case recursive case

--- Java -- cleaned doc text ---
Computes the factorial of n using recursion

## 6. Compute the documentation metrics for each language using `build.py`'s own functions

In [6]:
import pandas as pd


def report_metrics(label, doc_text_clean, code_text_no_doc):
    entropy = round(build.calculate_entropy(doc_text_clean), 4) if doc_text_clean else np.nan
    redundancy = round(build.doc_redundancy(doc_text_clean), 4) if doc_text_clean else np.nan
    overlap = round(build.doc_code_overlap(doc_text_clean, code_text_no_doc), 4) if doc_text_clean else np.nan
    readability = textstat.flesch_reading_ease(doc_text_clean) if doc_text_clean else np.nan
    return entropy, redundancy, overlap, readability


rows = []
for lang, result in extraction_results.items():
    entropy, redundancy, overlap, readability = report_metrics(
        lang, result["doc_cleaned"], result["code_text_no_doc"]
    )
    rows.append({
        "language": lang,
        "doc_entropy": entropy,
        "doc_redundancy": redundancy,
        "doc_code_overlap": overlap,
        "doc_readability": readability,
    })

metrics_df = pd.DataFrame(rows).set_index("language")
metrics_df


,doc_entropy,doc_redundancy,doc_code_overlap,doc_readability
language,,,,
Python,4.6861,0.2000,0.2857,50.238824
JavaScript,3.8797,0.2727,0.1250,31.006071
C#,3.9737,0.3200,0.1176,-3.175921
C/C++,3.7842,0.2500,0.2000,32.445132
Go,3.9321,0.1579,0.1250,36.245000
Java,3.7842,0.2500,0.2000,32.445132
Kotlin,3.8802,0.2381,0.1875,31.715000
PHP,3.8797,0.2727,0.1875,39.063214
Scala,3.7842,0.2500,0.1333,32.445132


## 7. Test `strip_comments` and `find_documentation_header` against known failure modes

Both functions were rewritten to use tree-sitter instead of regex/text
scanning, for the same reasons as `extract_documentation`. Neither had a
dedicated test section yet -- this verifies the specific bugs found and
fixed in each:

- `strip_comments`: a multi-line docstring was never removed (only
  same-line `"""..."""` was), any line starting with `#` was blindly
  treated as a comment (deleting real C preprocessor directives / Swift
  `#` macro calls), and a `//` inside a string literal (e.g. a URL)
  truncated the rest of that line.
- `find_documentation_header`: any line starting with `#`/`//` was
  blindly treated as a preceding comment header, regardless of language --
  wrongly attaching C preprocessor directives or Swift macro calls sitting
  above a function.

In [7]:
def check(label, condition):
    print(f"{'PASS' if condition else 'FAIL'} -- {label}")
    return condition

all_passed = True

# --- strip_comments ---

out = build.strip_comments('''def foo():
    """
    This is a multi-line docstring.
    It should be removed by strip_comments.
    """
    return 1
''', ".py")
all_passed &= check("strip_comments: multi-line docstring removed", '"""' not in out and "multi-line docstring" not in out)

out = build.strip_comments('''int foo() {
#ifdef DEBUG
    printf("debug mode");
#endif
    return 1;
}
''', ".c")
all_passed &= check("strip_comments: C preprocessor directive survives", "#ifdef DEBUG" in out and "#endif" in out)

out = build.strip_comments('''func test() {
    let x = 5
    #expect(x == 5)
}
''', ".swift")
all_passed &= check("strip_comments: Swift macro call survives", "#expect(x == 5)" in out)

out = build.strip_comments('''def foo():
    url = "https://example.com/path"
    return url
''', ".py")
all_passed &= check("strip_comments: URL with // inside a string literal survives intact", 'url = "https://example.com/path"' in out)

out = build.strip_comments('''def foo(x):
    """Docstring."""
    # a real comment
    return x  # inline comment
''', ".py")
all_passed &= check("strip_comments: real comments still removed", "Docstring" not in out and "a real comment" not in out and "inline comment" not in out and "return x" in out)

print()

# --- find_documentation_header ---

def adj(code, start_line, ext):
    lines = code.splitlines()
    return build.find_documentation_header(lines, start_line, ext)

all_passed &= check(
    "find_documentation_header: real comment attaches",
    adj('''# a real comment
def foo():
    return 1
''', 2, ".py") == 1
)

all_passed &= check(
    "find_documentation_header: C preprocessor directive does NOT attach",
    adj('''#include <stdio.h>
int foo() {
    return 1;
}
''', 2, ".c") == 2
)

all_passed &= check(
    "find_documentation_header: Swift macro call does NOT attach",
    adj('''#expect(x == 5)
func foo() -> Int {
    return 1
}
''', 2, ".swift") == 2
)

all_passed &= check(
    "find_documentation_header: comment attaches through <=5 blank lines",
    adj('''# a comment



def foo():
    return 1
''', 5, ".py") == 1
)

all_passed &= check(
    "find_documentation_header: comment does NOT attach through 6+ blank lines",
    adj('''# a comment






def foo():
    return 1
''', 8, ".py") == 8
)

print()
print("ALL PASSED" if all_passed else "SOME FAILED")


PASS -- strip_comments: multi-line docstring removed
PASS -- strip_comments: C preprocessor directive survives
PASS -- strip_comments: Swift macro call survives
PASS -- strip_comments: URL with // inside a string literal survives intact
PASS -- strip_comments: real comments still removed

PASS -- find_documentation_header: real comment attaches
PASS -- find_documentation_header: C preprocessor directive does NOT attach
PASS -- find_documentation_header: Swift macro call does NOT attach
PASS -- find_documentation_header: comment attaches through <=5 blank lines
PASS -- find_documentation_header: comment does NOT attach through 6+ blank lines

ALL PASSED


## 8. `find_documentation_header` in detail

Each case below shows the input code, the line the function actually
starts on, and exactly which lines get pulled into the "documentation
region" -- i.e. `lines[adj_start-1 : func_line-1]` -- so you can see
directly what gets attached and what doesn't, rather than just a
pass/fail boolean.

In [8]:
def show(label, code, func_line, ext):
    lines = code.splitlines()
    adj = build.find_documentation_header(lines, func_line, ext)
    print(f"=== {label} ({ext}) ===")
    print(f"function line: {func_line}  |  adjusted start: {adj}")
    if adj < func_line:
        print("Attached as documentation:")
        for i in range(adj, func_line):
            print(f"    {i}: {lines[i-1]!r}")
    else:
        print("Nothing attached -- function's own line is the start.")
    print(f"  --> {func_line}: {lines[func_line-1]!r}  (function itself)")
    print()


show("Single comment immediately above (attach)", '''# a real comment
def foo():
    return 1
''', 2, ".py")

show("Javadoc immediately above (attach)", '''/**
 * Computes something.
 * @param x input
 */
public int foo(int x) {
    return x;
}
''', 5, ".java")

show("Multiple consecutive // comments (attach all)", '''// line 1
// line 2
// line 3
func foo() -> Int {
    return 1
}
''', 4, ".swift")

show("Comment separated by 3 blank lines (attach, within tolerance of 5)", '''# a comment



def foo():
    return 1
''', 5, ".py")

show("Comment separated by 6 blank lines (NOT attached, exceeds tolerance)", '''# a comment






def foo():
    return 1
''', 8, ".py")

show("C preprocessor #include above (NOT attached -- real code, not a comment)", '''#include <stdio.h>
int foo() {
    return 1;
}
''', 2, ".c")

show("C preprocessor #ifdef above (NOT attached)", '''#ifdef DEBUG
int foo() {
    return 1;
}
#endif
''', 2, ".cpp")

show("Swift macro call above (NOT attached -- real executable code, not a comment)", '''#expect(x == 5)
func foo() -> Int {
    return 1
}
''', 2, ".swift")

show("Unrelated code directly above (NOT attached)", '''x = 5
def foo():
    return 1
''', 2, ".py")

show("Function at the very top of the file (no crash, nothing to attach)", '''def foo():
    return 1
''', 1, ".py")


=== Single comment immediately above (attach) (.py) ===
function line: 2  |  adjusted start: 1
Attached as documentation:
    1: '# a real comment'
  --> 2: 'def foo():'  (function itself)

=== Javadoc immediately above (attach) (.java) ===
function line: 5  |  adjusted start: 1
Attached as documentation:
    1: '/**'
    2: ' * Computes something.'
    3: ' * @param x input'
    4: ' */'
  --> 5: 'public int foo(int x) {'  (function itself)

=== Multiple consecutive // comments (attach all) (.swift) ===
function line: 4  |  adjusted start: 1
Attached as documentation:
    1: '// line 1'
    2: '// line 2'
    3: '// line 3'
  --> 4: 'func foo() -> Int {'  (function itself)

=== Comment separated by 3 blank lines (attach, within tolerance of 5) (.py) ===
function line: 5  |  adjusted start: 1
Attached as documentation:
    1: '# a comment'
    2: ''
    3: ''
    4: ''
  --> 5: 'def foo():'  (function itself)

=== Comment separated by 6 blank lines (NOT attached, exceeds tolerance) (.p

## 9. How many functions in the real dataset does this fix change?

Loads the actual mined dataset (`dataset/data/final_dataset_new_corrected.csv`,
13,809 functions, all 11 languages -- the same file behind the paper's
tables) and recomputes `doc_text` for every row using both the pre-fix and
post-fix `extract_documentation`, directly from the stored `function` column
(no repo re-cloning needed, since `function` already contains the full
source block the extraction runs over).

In [9]:
import pandas as pd

# Reconstruct the exact PRE-FIX (buggy) version for comparison
def old_extract_documentation(lines, start, end):
    raw_block = "\n".join(lines[start-1:end])
    extracted_docs = []
    c_blocks = re.findall(r'/\*.*?\*/', raw_block, flags=re.DOTALL)
    extracted_docs.extend([b.strip() for b in c_blocks])
    for line in raw_block.splitlines():
        line = line.strip()
        py_doc = re.findall(r'(""".*?"""|' + "'''.*?'''" + r'|"""[\s\S]*?"""|' + "'''[\\s\\S]*?''')", line)
        if py_doc:
            extracted_docs.extend(py_doc)
            continue
        comment_match = re.search(r'(//|#)(.*)$', line)
        if comment_match:
            extracted_docs.append(comment_match.group(0).strip())
    return extracted_docs, lines

df = pd.read_csv("dataset/data/final_dataset_new_corrected.csv")

ext_to_lang = {
    "cs": "C#", "c": "C", "cpp": "C++", "cc": "C++", "h": "C", "hpp": "C++",
    "cxx": "C++", "hxx": "C++", "go": "Go", "java": "Java", "js": "JavaScript",
    "jsx": "JavaScript", "kt": "Kotlin", "kts": "Kotlin", "php": "PHP",
    "py": "Python", "scala": "Scala", "swift": "Swift",
}
df["ext"] = df["file_path"].apply(lambda p: str(p).rsplit(".", 1)[-1].lower() if isinstance(p, str) and "." in p else "unknown")
df["language"] = df["ext"].map(ext_to_lang).fillna(df["ext"])

def recompute(func_text, ext):
    if not isinstance(func_text, str) or not func_text.strip():
        return "", ""
    lines = func_text.splitlines()
    old_doc, _ = old_extract_documentation(lines, 1, len(lines))
    new_doc, _ = build.extract_documentation(lines, 1, len(lines), "." + ext if ext else "")
    return " ".join(old_doc), " ".join(new_doc)

recomputed = df.apply(lambda row: recompute(row["function"], row["ext"]), axis=1)
df["old_doc_text"] = recomputed.apply(lambda t: t[0])
df["new_doc_text"] = recomputed.apply(lambda t: t[1])
df["changed"] = df["old_doc_text"] != df["new_doc_text"]

print(f"Total functions whose doc_text changes under the fix: {df['changed'].sum()} / {len(df)} ({100*df['changed'].mean():.2f}%)")
print()
print("Breakdown by language:")
summary = df.groupby("language").agg(total=("changed", "size"), changed=("changed", "sum"))
summary["pct"] = (100 * summary["changed"] / summary["total"]).round(2)
print(summary.sort_values("changed", ascending=False))


Total functions whose doc_text changes under the fix: 1600 / 13809 (11.59%)

Breakdown by language:
            total  changed    pct
language                         
Python       4211     1073  25.48
JavaScript   4392      256   5.83
Go            723       68   9.41
Swift         236       64  27.12
C++           814       63   7.74
C             515       33   6.41
PHP           595       23   3.87
Java         1481       10   0.68
C#            712        7   0.98
Scala          46        2   4.35
Kotlin         84        1   1.19


### Note: not every changed row is a pure bug fix

Inspecting a few non-Python diffs shows two different things happening:

1. **Python (~976 rows) and a handful of JS/C++/Java/PHP rows (~57)**: genuine
   fixes -- either the multi-line docstring is now captured at all (Python),
   or a duplicate `//`/`#` fragment that the old code double-counted from
   *inside* an already-captured `/* */` block (e.g. a URL like
   `https://lmms.io` inside a header comment, matched a second time as a
   spurious `//lmms.io` "comment") is no longer duplicated.
2. **A small number of Scala/Swift/C# rows**: the broadened triple-quote
   regex now also matches ordinary **raw/multi-line string literals** that
   have nothing to do with documentation -- Swift and C# both support
   `"""..."""` multi-line string literals as a language feature (not just
   Python-style docstrings), and Scala supports triple-quoted raw strings
   too. These get mistakenly captured as "documentation" by the widened
   regex. This is a new, narrow false-positive introduced by the fix, on top
   of the intended Python fix -- worth a follow-up refinement (e.g. only
   treating a triple-quoted block as a docstring when it is the first
   statement immediately after the function signature) if it matters for
   your results.